# Connecting to DB

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load key/value settings from .env in the workspace root.
load_dotenv()

host = os.getenv("AIVEN_PG_HOST")
port = os.getenv("AIVEN_PG_PORT", "22872")
user = os.getenv("AIVEN_PG_USER")
password = os.getenv("AIVEN_PG_PASSWORD")
database = os.getenv("AIVEN_PG_DATABASE")
sslmode = os.getenv("AIVEN_PG_SSLMODE", "require")

uri = (
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
    f"?sslmode={sslmode}"
)

engine = create_engine(uri, pool_pre_ping=True)

with engine.connect() as conn:
    row = conn.execute(text("SELECT current_database(), current_user")).fetchone()
    print(f"Connected to DB: {row[0]} as user: {row[1]}")

Connected to DB: sponsor_db as user: avnadmin


# Read Data

### Read df_companies

In [ ]:
import pandas as pd

df_companies = pd.read_csv("data_source/companies.csv")
df_companies.head()

,Rank,Name,Symbol,marketcap,price (HKD),country
0,1,AIA,1299.HK,8.829573e+11,84.400000,Hong Kong
1,2,Hong Kong Exchanges & Clearing,0388.HK,5.141575e+11,406.800000,Hong Kong
2,3,Bank of China (Hong Kong),2388.HK,4.444797e+11,42.040000,Hong Kong
3,4,Jardine Matheson,J36.SI,4.296833e+11,595.954582,Hong Kong
4,5,Sun Hung Kai Properties,0016.HK,4.126439e+11,142.400000,Hong Kong


### Read df_data_company

In [ ]:
df_data_company = pd.read_json("data_source/data_company.json")

In [5]:
import ast
import re

def _to_dict(x):
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return {}
    try:
        return ast.literal_eval(str(x))
    except (ValueError, SyntaxError):
        return {}

def _clean_english(x):
    if pd.isna(x):
        return x
    x = re.sub(r"[^\x00-\x7F]+", "", str(x))  # keep ASCII only
    return re.sub(r"\s+", " ", x).strip()

# 1) Split email_phone into email and phone
parsed_contact = df_data_company["email_phone"].apply(_to_dict)
df_data_company["email"] = parsed_contact.apply(lambda d: d.get("email", ""))
df_data_company["phone"] = parsed_contact.apply(lambda d: d.get("phone", ""))
df_data_company = df_data_company.drop(columns=["email_phone"])

# 2) Convert company_size to integer
df_data_company["company_size"] = (
    df_data_company["company_size"]
    .astype(str)
    .str.extract(r"([\d,]+)", expand=False)
    .str.replace(",", "", regex=False)
    .astype("Int64")
)

# Remove non-English characters from all text columns
text_cols = df_data_company.select_dtypes(include="object").columns
df_data_company[text_cols] = df_data_company[text_cols].apply(lambda col: col.map(_clean_english))

df_data_company.head()

,company_name,contact_person,job_title,website,company_size,industry,email,phone
0,AIA Group Limited,Group Corporate Communications (Media) / Inves...,Corporate Communications / Investor Relations ...,https://www.aia.com,25938,Insurance,mediarelations@aia.com / ir@aia.com,+852 2232 8888 (Hong Kong Customer Hotline)
1,Hong Kong Exchanges and Clearing Limited (HKEX),Group Corporate Communications (Media) / Inves...,Corporate Communications / Investor Relations ...,https://www.hkex.com.hk,2423,Financial Services (Stock Exchange & Clearing),info@hkex.com.hk / investorrelations@hkex.com.hk,+852 2522 1122 / +852 2840 3330 (Enquiry Hotline)
2,Bank of China (Hong Kong) Limited (BOCHK),Group Corporate Communications (Media) / Inves...,Corporate Communications / Investor Relations ...,https://www.bochk.com,15228,Banking and Financial Services,corp_comm@bochk.com / investor_relations@bochk...,+852 3988 2388 (Personal Customer Hotline) / +...
3,Jardine Matheson Holdings Limited (),Lincoln Pan / Group Corporate Communications,Group Executive Chairman and Managing Director...,https://www.jardines.com,443000,Diversified Conglomerate (),jml@jardines.com / gc@jardines.com,+852 2843 8288
4,Sun Hung Kai Properties Limited (),Raymond Kwok (Chairman) / Media Relations Team,Chairman and Managing Director (Raymond Kwok),https://www.shkp.com,38000,Real Estate Development and Investment,shkp@shkp.com / media@shkp.com,+852 2827 8111 (General) / +852 3766 5787 (Inv...


### Read df_staff_related

In [12]:
df_staff_related = pd.read_json("data_source/data_staff_related.json")

In [14]:
def _split_email_phone(value):
    if pd.isna(value):
        return pd.Series({"email": "", "phone": ""})

    if isinstance(value, dict):
        d = value
    else:
        s = str(value).strip()
        if s in {"", "N/A", "NA", "None", "null"}:
            return pd.Series({"email": "", "phone": ""})
        try:
            d = ast.literal_eval(s) if s.startswith("{") and s.endswith("}") else {}
        except (ValueError, SyntaxError):
            d = {}

        if not d:
            return pd.Series({
                "email": s if "@" in s else "",
                "phone": "" if "@" in s else s
            })

    email = (
        d.get("email")
        or d.get("general_email")
        or d.get("media_email")
        or d.get("investor_relations_email")
        or ""
    )
    phone = (
        d.get("phone")
        or d.get("general_phone")
        or d.get("media_phone")
        or ""
    )

    return pd.Series({"email": str(email), "phone": str(phone)})

df_staff_related[["email", "phone"]] = df_staff_related["email_phone"].apply(_split_email_phone)
df_staff_related = df_staff_related.drop(columns=["email_phone"])

df_staff_related.head()

,Name,Title,linkedin_url,address,company_name,email,phone
0,Melissa Wong,Chief Customer & Marketing Officer,https://www.linkedin.com/in/melissa-wong-4ba0374,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong and Macau,,
1,Alger Fung,Chief Executive Officer,https://www.linkedin.com/in/alger-fung-11531210b,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong and Macau,,
2,Stuart A. Spencer,Chief Marketing Officer,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Group,,
3,Amy Lo,Head and Chief Executive,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong,,
4,Esther Chan,Communications Manager,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong,,+852 2100 1416


# Update the Database

### Create Table Schemas

In [ ]:
# # Create table schemas for the three dataframes
# ddl_statements = [
#     """
#     CREATE TABLE IF NOT EXISTS companies_csv (
#         "Rank" BIGINT,
#         "Name" TEXT,
#         "Symbol" TEXT,
#         marketcap DOUBLE PRECISION,
#         "price (HKD)" DOUBLE PRECISION,
#         country TEXT
#     );
#     """,
#     """
#     CREATE TABLE IF NOT EXISTS data_company (
#         company_name TEXT,
#         contact_person TEXT,
#         job_title TEXT,
#         website TEXT,
#         company_size BIGINT,
#         industry TEXT,
#         email TEXT,
#         phone TEXT
#     );
#     """,
#     """
#     CREATE TABLE IF NOT EXISTS staff_related (
#         "Name" TEXT,
#         "Title" TEXT,
#         linkedin_url TEXT,
#         address TEXT,
#         company_name TEXT,
#         email TEXT,
#         phone TEXT
#     );
#     """
# ]

# with engine.begin() as conn:
#     for ddl in ddl_statements:
#         conn.execute(text(ddl))

# print("Schemas created (or already existed): companies_csv, data_company, staff_related")

Schemas created (or already existed): companies_csv, data_company, staff_related


### Check the tables in DB

In [17]:
# List user tables in the current PostgreSQL database
with engine.connect() as c:
    tables = c.execute(text("""
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE table_type = 'BASE TABLE'
          AND table_schema NOT IN ('pg_catalog', 'information_schema')
        ORDER BY table_schema, table_name
    """)).fetchall()

pd.DataFrame(tables, columns=["schema", "table_name"])

,schema,table_name
0,public,companies_csv
1,public,data_company
2,public,staff_related


### Delete and Re-import the df data into the db tables

In [18]:
# Delete existing rows and re-import all dataframe data into PostgreSQL tables
with engine.begin() as conn:
    conn.execute(text("""
        TRUNCATE TABLE companies_csv, data_company, staff_related;
    """))

    df_companies.to_sql("companies_csv", con=conn, if_exists="append", index=False, method="multi")
    df_data_company.to_sql("data_company", con=conn, if_exists="append", index=False, method="multi")
    df_staff_related.to_sql("staff_related", con=conn, if_exists="append", index=False, method="multi")

# Quick validation
pd.read_sql("""
    SELECT 'companies_csv' AS table_name, COUNT(*) AS row_count FROM companies_csv
    UNION ALL
    SELECT 'data_company', COUNT(*) FROM data_company
    UNION ALL
    SELECT 'staff_related', COUNT(*) FROM staff_related
    ORDER BY table_name
""", con=engine)

,table_name,row_count
0,companies_csv,151
1,data_company,5
2,staff_related,25


In [21]:
pd.read_sql("SELECT * FROM staff_related LIMIT 5", con=engine)

,Name,Title,linkedin_url,address,company_name,email,phone
0,Melissa Wong,Chief Customer & Marketing Officer,https://www.linkedin.com/in/melissa-wong-4ba0374,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong and Macau,,
1,Alger Fung,Chief Executive Officer,https://www.linkedin.com/in/alger-fung-11531210b,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong and Macau,,
2,Stuart A. Spencer,Chief Marketing Officer,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Group,,
3,Amy Lo,Head and Chief Executive,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong,,
4,Esther Chan,Communications Manager,N/A,"AIA Central, 1 Connaught Road Central, Hong Kong",AIA Hong Kong,,+852 2100 1416
